# 1. Estimativa de Pose com YOLOv8 (Pose Estimation)

## 1.1 Objetivo
Este notebook explora a tarefa de **Estimativa de Pose** (Pose Estimation). Diferente da detecção de objetos comum (que encontra caixas), este modelo identifica **pontos-chave** (keypoints) no corpo humano, como articulações (cotovelos, joelhos, ombros), permitindo análise biomecânica e de movimentos.

## 1.2 O que são Keypoints?
Keypoints são coordenadas (x, y) específicas que representam partes anatômicas de interesse. O modelo YOLOv8-Pose padrão é treinado no dataset COCO, que define **17 pontos** principais para descrever o esqueleto humano.

# 2. Primeiros Passos

Vamos carregar o modelo específico para pose (`yolo11n-pose.pt`) e testá-lo em um vídeo estático.

In [ ]:
from ultralytics import YOLO

# Carregar o modelo YOLO pré-treinado para estimativa de pose
model = YOLO("yolo11n-pose.pt")

# Caminho do vídeo
source = 'original/video-pose.mp4'

# Realizar inferência e salvar o resultado
results = model(source, save=True, conf=0.5)

# 3. Entendendo os Resultados (Keypoints)

## 3.1 O Objeto Pose
Os resultados da inferência ficam em `results[0].keypoints`. Este objeto contém:
- `xy`: Coordenadas x, y de cada ponto.
- `conf`: Confiança da detecção de cada ponto.
- `xyn`: Coordenadas normalizadas (entre 0 e 1).

## 3.2 Tabela de Referência dos Keypoints (COCO)
Para trabalhar com os dados, você precisa saber qual índice corresponde a qual parte do corpo. Use a tabela abaixo como referência:

| ID | Parte do Corpo | ID | Parte do Corpo |
|---|---|---|---|
| 0 | Nariz (Nose) | 9 | Punho Esquerdo |
| 1 | Olho Esquerdo | 10 | Punho Direito |
| 2 | Olho Direito | 11 | Quadril Esquerdo |
| 3 | Orelha Esquerda | 12 | Quadril Direito |
| 4 | Orelha Direita | 13 | Joelho Esquerdo |
| 5 | Ombro Esquerdo | 14 | Joelho Direito |
| 6 | Ombro Direito | 15 | Tornozelo Esquerdo |
| 7 | Cotovelo Esquerdo | 16 | Tornozelo Direito |
| 8 | Cotovelo Direito | | |

# 4. Detecção em Tempo Real (Webcam)

## 4.1 Visualizando o Esqueleto
Assim como na detecção de objetos, podemos usar a webcam. O método `plot()` desenhará automaticamente o esqueleto conectando os pontos.

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO('yolo11n-pose.pt')
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erro ao acessar webcam")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    # O plot() desenha os keypoints e o esqueleto
    annotated_frame = results[0].plot()

    cv2.imshow("YOLOv8 Pose", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# 5. Aplicação Avançada: Introdução à Análise de Movimento

A grande utilidade do Pose Estimation é calcular a relação entre os pontos. Por exemplo, calcular o **ângulo** entre o ombro, cotovelo e punho pode dizer se um braço está esticado ou flexionado — a base para contadores de repetições em exercícios.

## 5.1 Matemática dos Ângulos
Podemos usar a função arco-tangente (`atan2`) para encontrar o ângulo entre três pontos.

In [ ]:
import math

def calcular_angulo(p1, p2, p3):
    """
    Calcula o ângulo entre três pontos (x, y).
    p2 é o ponto central (vértice).
    """
    # Desempacotar coordenadas
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3

    # Calcular ângulo usando atan2
    angulo = math.degrees(math.atan2(y3 - y2, x3 - x2) - math.atan2(y1 - y2, x1 - x2))
    
    # Garantir que o ângulo seja positivo
    if angulo < 0:
        angulo += 360

    return angulo

## 5.2 Exemplo Prático: Monitorando o Braço Direito
O código abaixo acessa os keypoints específicos do braço direito:
- Ombro Direito (ID 6)
- Cotovelo Direito (ID 8)
- Punho Direito (ID 10)

Ele calcula o ângulo e o exibe na tela.

In [ ]:
import cv2
import math
from ultralytics import YOLO

# Função auxiliar (redeclarada para garantir que esteja no contexto)
def calcular_angulo(p1, p2, p3):
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    angulo = math.degrees(math.atan2(y3 - y2, x3 - x2) - math.atan2(y1 - y2, x1 - x2))
    if angulo < 0: angulo += 360
    return angulo

model = YOLO('yolo11n-pose.pt')
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break

    results = model(frame, verbose=False) # verbose=False para limpar o terminal
    
    # Desenhar esqueleto padrão
    frame = results[0].plot()

    # Acessar Keypoints
    # results[0].keypoints.xy é um tensor [N, 17, 2]. 
    # Pegamos a primeira pessoa [0] (se houver).
    if len(results[0].keypoints.xy) > 0:
        keypoints = results[0].keypoints.xy[0]

        # IDs: 6 (Ombro), 8 (Cotovelo), 10 (Punho) -> Lado DIREITO
        # Verificando se a confiança (conf) é boa o suficiente antes de calcular
        # Nota: keypoints[i] retorna [x, y]
        
        ombro = keypoints[6]
        cotovelo = keypoints[8]
        punho = keypoints[10]

        # Verifica se nenhum ponto é (0,0) - indicando não detecção
        if ombro[0] > 0 and cotovelo[0] > 0 and punho[0] > 0:
            # Converter tensores para lista python para a função matemática
            p_ombro = (float(ombro[0]), float(ombro[1]))
            p_cotovelo = (float(cotovelo[0]), float(cotovelo[1]))
            p_punho = (float(punho[0]), float(punho[1]))

            angulo = calcular_angulo(p_ombro, p_cotovelo, p_punho)

            # Exibir o ângulo perto do cotovelo
            cv2.putText(frame, f"{int(angulo)} graus", 
                        (int(p_cotovelo[0]), int(p_cotovelo[1] - 20)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow("Analise de Angulo", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# 6. Conclusão

## 6.1 Resumo
Vimos como o YOLOv8-Pose vai além de caixas delimitadoras, fornecendo uma compreensão detalhada da postura corporal. Com os 17 keypoints mapeados e um pouco de trigonometria, podemos criar aplicações complexas de análise de movimento.

## 6.2 Próximos Passos
- Tente calcular ângulos para as pernas (Quadril-Joelho-Tornozelo) usando os IDs da tabela.
- Crie um contador de repetições: Se ângulo < 30 (flexionado) e depois > 160 (esticado) = 1 repetição.